In [ ]:
import re
import pandas as pd

In [ ]:
def coordinates_to_lat_long(coord_str):
    lat, long = coord_str.split(',')
    return float(lat), float(long)

def parse_owners(owner_str):
    if pd.isna(owner_str) or str(owner_str).strip() == '-':
        return {}
    
    # Try to find name [XX%] pairs
    matches = re.findall(r'([^;]+?)\s*\[(\d+(?:\.\d+)?)%\]', owner_str)
    
    if matches:
        # Sort by percentage descending
        matches = sorted(matches, key=lambda x: float(x[1]), reverse=True)
        result = {}
        ranks = ['Primary', 'Secondary', 'Third', 'Fourth', 'Fifth']
        for i, (name, pct) in enumerate(matches):
            label = ranks[i] if i < len(ranks) else f'Owner_{i+1}'
            result[f'{label} Project Owner'] = name.strip()
            result[f'{label} Ownership (%)'] = float(pct)
        return result
    else:
        # No percentages — treat whole string as primary owner
        return {'Primary Project Owner': owner_str.strip(),
                'Primary Ownership (%)': 100.0 } # ← only one name, so assumed 100%
    
def parse_ownersID(owner_str):
    if pd.isna(owner_str) or str(owner_str).strip() == '-':
        return {}
    
    # Try to find name [XX%] pairs
    matches = re.findall(r'([^;]+?)\s*\[(\d+(?:\.\d+)?)%\]', owner_str)
    
    if matches:
        # Sort by percentage descending
        matches = sorted(matches, key=lambda x: float(x[1]), reverse=True)
        result = {}
        ranks = ['Primary', 'Secondary', 'Third', 'Fourth', 'Fifth']
        for i, (name, pct) in enumerate(matches):
            label = ranks[i] if i < len(ranks) else f'Owner_{i+1}'
            result[f'{label} Project Owner ID'] = name.strip()
        return result
    else:
        # No percentages — treat whole string as primary owner
        return {'Primary Project Owner ID': owner_str.strip()} # ← only one name, so assumed 100%

In [69]:
def global_iron_steel_tracker_data_analysis(iron_steel_property, iron_steel_owners):
    # Separate GPS into two columns, Latitude and Longitude
    iron_steel_property[['Lat', 'Lon']] = iron_steel_property['Coordinates'].apply(lambda x: pd.Series(coordinates_to_lat_long(x)))

    # Parse the Parent column to extract up to 5 owners and their ownership percentages, and add these as new columns to the dataframe
    owner_cols = iron_steel_property['Parent (English)'].apply(lambda x: pd.Series(parse_owners(x)))
    owner_cols_ID = iron_steel_property['Parent GEM Entity ID'].apply(lambda x: pd.Series(parse_ownersID(x)))
    iron_steel_property = pd.concat([iron_steel_property, owner_cols, owner_cols_ID], axis=1)

    # Loop over each xxx project parent ID column, for each column, loop over each row, find the country name in the iron_steel_owners dataframe using the Parent GEM Entity ID that matches the ID in each row, and add a new column with the country name for each owner column. 
    # If the row is blank, leave it alone, if there is no match between the two dataframe, leave the cell value blank. There may be multiple matches in the iron_steel_owners dataframe for each owner ID, but we only keep the first matched country name for each owner ID.
    # The drop column has the same name 'Parent GEM Entity ID' as the merge key, so drop it after each merge to avoid duplicate columns in the dataframe
    hq_lookup = (
        iron_steel_owners
        .drop_duplicates(subset='Parent GEM Entity ID')
        .set_index('Parent GEM Entity ID')['Parent Headquarters Country']
        .to_dict()
    )

    for col in list(iron_steel_property.columns):  # list() captures columns before loop modifies them
        if col.endswith('Project Owner ID'):
            rank = col.replace(' Project Owner ID', '')
            iron_steel_property[f'{rank} Owner Country'] = iron_steel_property[col].map(hq_lookup)


    # Drop columns
    iron_steel_property = iron_steel_property.drop(columns=['Other plant names (English)','Other plant names (other language)', 'Owner (other language)', 'Owner GEM Entity ID', 'Owner PermID', 'SOE status',\
                                                    'Parent PermID','Location address,'	'Location address (other language)','Coordinate accuracy','Plant age', 'Announced date','Construction date','Start date'\
                                                    'Pre-retirement announcement date',	'Idled date',	'Retired date',	'Ferronickel capacity (ttpa)',	'Sinter plant capacity (ttpa)',	'Coking plant capacity (ttpa)',\
                                                    'Pelletizing plant capacity (ttpa)',	'Category steel product',	'Steel products',	'Steel sector end users', 'ISO 14001',	'ISO 50001',	'ResponsibleSteel certification',\
                                                    'Power source', 'Met coal source'])

    # Reorder columns: put 'Lat' and 'Lon' after 'Coordinates', put 'GEM wiki page URL' to the last column
    # Update column name: 'Municipality' to 'City', 'Subnational unit' to 'State/Province', 'Country/Area' to 'Country'
    cols = list(iron_steel_property.columns)
    cols.insert(cols.index('Coordinates') + 1, cols.pop(cols.index('Lat'))) 
    cols.insert(cols.index('Coordinates') + 1, cols.pop(cols.index('Lon')))
    cols.append(cols.pop(cols.index('GEM wiki page URL')))  
    cols[cols.index('Municipality')] = 'City'
    cols[cols.index('Subnational unit')] = 'State/Province'
    cols[cols.index('Country/Area')] = 'Country'
    iron_steel_property = iron_steel_property[cols] 

    return iron_steel_property


def capacity_production_merge(iron_steel_capacity, iron_steel_production):
    # Process capacity dataset==============================================================================================================
    # Filter the iron_steel_capacity dataframe based on status
    iron_steel_capacity = iron_steel_capacity[iron_steel_capacity['Status'].isin(['operating', 'operating pre-retirement', 'announced', 'mothballed', 'construction'])]
    # Change the status name 
    iron_steel_capacity['Status'] = iron_steel_capacity['Status'].replace({
        'announced': 'probable',        
        'operating pre-retirement': 'operating',
        'mothballed': 'highly probable',
        'construction': 'highly probable'
    })
    EQUIP_TO_CAP = {
        'EAF':                     'Nominal EAF steel capacity (ttpa)',
        'BOF':                     'Nominal BOF steel capacity (ttpa)',
        'IF':                      'Nominal IF steel capacity (ttpa)',
        'Steel other/unspecified': 'Other/unspecified steel capacity (ttpa)',
        'BF':                      'Nominal BF capacity (ttpa)',
        'DRI':                     'Nominal DRI capacity (ttpa)',
        'Iron other/unspecified':  'Other/unspecified iron capacity (ttpa)',
    }

    # Split equipment strings into lists, then explode to one row per equipment
    df_melted = iron_steel_capacity.copy()
    df_melted['equipment_type'] = df_melted['Main production equipment'].str.split('; ')
    df_melted = df_melted.explode('equipment_type').reset_index(drop=True)

    # Pull the matching capacity value for each equipment row
    df_melted['capacity_ttpa'] = df_melted.apply(
        lambda r: r[EQUIP_TO_CAP[r['equipment_type']]]
                if pd.notna(r['equipment_type']) and r['equipment_type'] in EQUIP_TO_CAP
                else pd.NA,
        axis=1
    )

    # Drop all individual capacity columns, the original combined equipment column,
    # and the two aggregate columns
    drop_cols = list(EQUIP_TO_CAP.values()) + [
        'Main production equipment',
        'Nominal crude steel capacity (ttpa)',
        'Nominal iron capacity (ttpa)',
        'Plant name (other language)',
        'Country/area',
        'Start date'
    ]
    df_melted = df_melted.drop(columns=drop_cols)

    print(f'Original rows: {len(iron_steel_capacity)}, Melted rows: {len(df_melted)}')
    print(f'Equipment types: {df_melted["equipment_type"].unique()}')
    # df_melted.head(10)
#==================================================================================================    
    # Process production dataset
    YEARS = [2025, 2024, 2023, 2022, 2021, 2020, 2019]

    def most_recent_production(row):
        for yr in YEARS:
            val = row[yr]
            if val == '>0':
                return 0          # treat as minimal production
            if val != 'unknown' and pd.notna(val):
                try:
                    return float(val)
                except (ValueError, TypeError):
                    continue
        return 0


    iron_steel_production['production_ttpa'] = iron_steel_production.apply(most_recent_production, axis=1)

    # Remove the string starting from 'production' for each row in the 'Type of production' column, and assign the remaining string to a new column called 'production_type'
    iron_steel_production['equipment_type'] = iron_steel_production['Type of production'].str.replace('production (ttpa)', '').str.strip()

    # Remove rows where 'production_type' starts with 'Crude' or 'Iron
    iron_steel_production = iron_steel_production[~iron_steel_production['equipment_type'].str.startswith(('Crude', 'Iron'), na=False)]

    # Change the euqipment+_type name 'Other/unspecified steel' to 'Steel other/unspecified', and 'Other/unspecified iron' to 'Iron other/unspecified' to match the equipment_type name in the capacity dataset
    # change the equipment_type name 'EAF steel' to 'EAF', 'BOF steel' to 'BOF', 'IF steel' to 'IF', 'BF iron' to 'BF', and 'DRI iron' to 'DRI' to match the equipment_type name in the capacity dataset
    iron_steel_production['equipment_type'] = iron_steel_production['equipment_type'].replace({
        'Other/Unknown steel': 'Steel other/unspecified', 
        'Other/Unknown iron': 'Iron other/unspecified',
        'EAF steel': 'EAF',
        'BOF steel': 'BOF',
        'IF steel': 'IF',
        'BF iron': 'BF',
        'DRI iron': 'DRI'
    })

    # Drop year columns, Type of production, and Plant name
    iron_steel_production = iron_steel_production.drop(columns=YEARS + ['Type of production'])
    
    # Merge df_melted and iron_steel_production dataframes on 'GEM plant ID' and 'equipment_type' (left join)
    merged_df = pd.merge(df_melted, iron_steel_production[['GEM plant ID', 'equipment_type', 'production_ttpa']],
                         on=['GEM plant ID', 'equipment_type'], how='left')
    # Change N/A production_ttpa to 0
    merged_df['production_ttpa'] = merged_df['production_ttpa'].fillna(0)
    merged_df.drop(columns=['equipment_type'], inplace=True)  # drop the equipment_type column after merge
    merged_df.drop_duplicates(inplace=True)  # drop duplicates if any

    return merged_df




In [18]:
# Load data
iron_mine_data = pd.read_excel('../data/Processed_data/iron_mine_w_cost_FeContent.xlsx')
iron_steel_property = pd.read_excel('../data/GlobalTrackerData/steel/Plant-level-data-Global-Iron-and-Steel-Tracker-March-2026-V1.xlsx', sheet_name='Plant data')
iron_steel_capacity = pd.read_excel('../data/GlobalTrackerData/steel/Plant-level-data-Global-Iron-and-Steel-Tracker-March-2026-V1.xlsx', sheet_name='Plant capacities and status')
iron_steel_production = pd.read_excel('../data/GlobalTrackerData/steel/Plant-level-data-Global-Iron-and-Steel-Tracker-March-2026-V1.xlsx', sheet_name='Plant production')
iron_steel_owners = pd.read_excel('../data/GlobalTrackerData/steel/Global-Energy-Ownership-Tracker-March-2026-V1.xlsx', sheet_name='Steel Plant Ownership')
# Create another dataframe with exactly the same columns as iron_mine_data
final_df = pd.DataFrame(columns=iron_mine_data.columns)

# iron_steel_property_subset = global_iron_steel_tracker_data_analysis(iron_steel_property, iron_steel_owners)

In [72]:
cap_production_merged = capacity_production_merge(iron_steel_capacity, iron_steel_production)
print(cap_production_merged.shape)
print(cap_production_merged.head(10))

Original rows: 1534, Melted rows: 2326
Equipment types: <StringArray>
[                    'EAF',                     'DRI',
                      'BF',                     'BOF',
                      'IF', 'Steel other/unspecified',
  'Iron other/unspecified']
Length: 7, dtype: str
(2274, 5)
    GEM plant ID                           Plant name (English)  \
0  P100000120882                 Aba Iron and Steel Payas plant   
1  P100000120753               Abba Steel Ohangwena steel plant   
2  P100000120802                    Abinsk Electric Steel Works   
3  P100000120020               Abul Khair Steel Sitakunda plant   
4  P100000120620        Acciaierie d'Italia Taranto steel plant   
5  P100000120620        Acciaierie d'Italia Taranto steel plant   
6  P100000120620        Acciaierie d'Italia Taranto steel plant   
7  P100000120620        Acciaierie d'Italia Taranto steel plant   
8  P100000120634  Acciaierie Venete Borgo Valsugana steel plant   
9  P100000120636            Acciaie